# NB06 — P2P Node Protocol

Build a real (toy) Ethereum-style peer protocol over TCP.
Nodes connect to each other, handshake, gossip transactions using sqrt-fanout.

**Dependencies:** `_lib/framing.py`, `_lib/keccak.py` (from prior notebooks).


## 1. From client-server to peer-to-peer

NB05 had a server and a client. In P2P, every node is BOTH.
Each node listens for incoming peers AND dials out to known peers.
We will build a `Node` class that does both, then run 8 nodes locally
and watch a transaction propagate across the mesh.

Key ideas:

* **Dual role** — every node binds a listen port AND connects to peers.
* **Handshake** — nodes exchange `HELLO` (id + port) before sending anything else.
* **Gossip** — when a node learns a new tx it forwards it; no central coordinator.
* **Sqrt-fanout** — the `eth/68` pattern: send the full body to ~√N peers,
  send only the hash (`ANNOUNCE_TX`) to the rest. Recipients that want the body
  request it with `GET_TX`. This bounds bandwidth at scale.


## 2. Message types

| Type | Code | Payload |
|------|------|---------|
| HELLO | `0x01` | `node_id (8 bytes) \|\| port (2 bytes BE)` |
| ANNOUNCE_TX | `0x04` | `tx_hash (32 bytes)` |
| GET_TX | `0x05` | `tx_hash (32 bytes)` |
| TX | `0x06` | `tx_bytes (arbitrary)` |

Codes `0x02` (GETPEERS) and `0x03` (PEERS) are reserved for a future session.

Every message is wrapped in the 4-byte length-prefix framing from `_lib/framing.py`.


In [1]:
import struct

# Message type constants
HELLO       = 0x01
ANNOUNCE_TX = 0x04
GET_TX      = 0x05
TX          = 0x06

def encode_msg(msg_type: int, body: bytes) -> bytes:
    """Prepend the single-byte type code."""
    return bytes([msg_type]) + body

def decode_msg(payload: bytes) -> tuple:
    """Split (msg_type, body)."""
    return payload[0], payload[1:]

print('constants:', hex(HELLO), hex(ANNOUNCE_TX), hex(GET_TX), hex(TX))


constants: 0x1 0x4 0x5 0x6


## 3. The `Node` class

The `Node` class lives in `_lib/peer.py`.  It combines:

* A **listener thread** that `accept()`s inbound connections
  (with a 0.5 s socket timeout so it checks `_stop` frequently).
* A **handshake** that exchanges `HELLO` before any data flows.
* A **per-peer reader thread** that handles `ANNOUNCE_TX`, `GET_TX`, and `TX`.
* A **`stop()`** method that sets `_stop`, closes all peer sockets, and
  closes the listener — letting every daemon thread exit cleanly.

The full source is written to disk with `%%writefile` in the next cell.


In [2]:
%%writefile _lib/peer.py
"""peer.py — toy Ethereum-style P2P node over TCP.

Message types
-------------
HELLO        0x01  — node_id (8 bytes) || port (2 bytes BE)
ANNOUNCE_TX  0x04  — tx_hash (32 bytes)
GET_TX       0x05  — tx_hash (32 bytes)
TX           0x06  — tx_bytes (arbitrary)

Codes 0x02 (GETPEERS) and 0x03 (PEERS) are reserved for a future session.
"""

import math
import random
import secrets
import socket
import struct
import threading

from .framing import send_msg, recv_msg
from .keccak import keccak256

# ---------------------------------------------------------------------------
# Message type constants
# ---------------------------------------------------------------------------

HELLO = 0x01
ANNOUNCE_TX = 0x04
GET_TX = 0x05
TX = 0x06


# ---------------------------------------------------------------------------
# Encode / decode helpers
# ---------------------------------------------------------------------------

def encode_msg(msg_type: int, body: bytes) -> bytes:
    """Prepend the single-byte type code to *body*."""
    return bytes([msg_type]) + body


def decode_msg(payload: bytes) -> tuple:
    """Split a received payload into (msg_type, body)."""
    return payload[0], payload[1:]


# ---------------------------------------------------------------------------
# Node class
# ---------------------------------------------------------------------------

class Node:
    """A dual-role (listener + dialer) P2P node.

    Each node:
    * Listens on *port* for inbound peers.
    * Can dial out to known peers via :meth:`connect`.
    * Gossips new transactions using sqrt-fanout (the eth/68 pattern).
    """

    def __init__(self, port: int):
        self.port = port
        self.node_id = secrets.token_bytes(8)
        self.peers: dict = {}  # peer_id -> socket
        self.seen_tx: set = set()
        self.tx_pool: dict = {}  # hash -> raw bytes
        self.lock = threading.Lock()
        self._stop = threading.Event()
        self._listener_sock = None

    # ------------------------------------------------------------------
    # Lifecycle
    # ------------------------------------------------------------------

    def start(self):
        """Start the background listener thread."""
        threading.Thread(target=self._listen, daemon=True).start()

    def stop(self):
        """Signal all threads to exit and close all sockets."""
        self._stop.set()
        with self.lock:
            for pid, conn in list(self.peers.items()):
                try:
                    conn.shutdown(socket.SHUT_RDWR)
                except OSError:
                    pass
                try:
                    conn.close()
                except OSError:
                    pass
            self.peers.clear()
        if self._listener_sock is not None:
            try:
                self._listener_sock.close()
            except OSError:
                pass

    # ------------------------------------------------------------------
    # Listener
    # ------------------------------------------------------------------

    def _listen(self):
        srv = socket.socket()
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(("127.0.0.1", self.port))
        srv.listen(20)
        srv.settimeout(0.5)  # allows _stop checks
        self._listener_sock = srv
        while not self._stop.is_set():
            try:
                conn, _ = srv.accept()
            except socket.timeout:
                continue
            except OSError:
                break
            threading.Thread(
                target=self._handshake_inbound,
                args=(conn,),
                daemon=True,
            ).start()

    # ------------------------------------------------------------------
    # Handshake — inbound direction
    # ------------------------------------------------------------------

    def _handshake_inbound(self, conn):
        try:
            conn.settimeout(2.0)
            payload = recv_msg(conn)
            mt, body = decode_msg(payload)
            if mt != HELLO:
                conn.close()
                return
            peer_id = body[:8]
            peer_port = struct.unpack(">H", body[8:10])[0]
            with self.lock:
                if peer_id == self.node_id or peer_id in self.peers:
                    conn.close()
                    return
                self.peers[peer_id] = conn
            send_msg(conn, encode_msg(HELLO, self.node_id + struct.pack(">H", self.port)))
            print(f"[{self.port}] inbound peer {peer_id.hex()} from :{peer_port}")
            self._peer_loop(peer_id, conn)
        except Exception:
            conn.close()

    # ------------------------------------------------------------------
    # Handshake — outbound direction
    # ------------------------------------------------------------------

    def connect(self, host: str, port: int) -> bool:
        """Dial *host:port*, exchange HELLO, and start a peer loop."""
        try:
            conn = socket.socket()
            conn.settimeout(2.0)
            conn.connect((host, port))
            send_msg(conn, encode_msg(HELLO, self.node_id + struct.pack(">H", self.port)))
            payload = recv_msg(conn)
            mt, body = decode_msg(payload)
            if mt != HELLO:
                conn.close()
                return False
            peer_id = body[:8]
            with self.lock:
                if peer_id == self.node_id or peer_id in self.peers:
                    conn.close()
                    return False
                self.peers[peer_id] = conn
            print(f"[{self.port}] outbound peer {peer_id.hex()} -> :{port}")
            threading.Thread(
                target=self._peer_loop,
                args=(peer_id, conn),
                daemon=True,
            ).start()
            return True
        except (OSError, ConnectionError):
            return False

    # ------------------------------------------------------------------
    # Per-peer message loop
    # ------------------------------------------------------------------

    def _peer_loop(self, peer_id: bytes, conn: socket.socket):
        try:
            while not self._stop.is_set():
                try:
                    payload = recv_msg(conn)
                except socket.timeout:
                    continue
                mt, body = decode_msg(payload)
                if mt == ANNOUNCE_TX:
                    self._handle_announce(peer_id, body)
                elif mt == GET_TX:
                    with self.lock:
                        tx_body = self.tx_pool.get(body)
                    if tx_body is not None:
                        try:
                            send_msg(conn, encode_msg(TX, tx_body))
                        except OSError:
                            pass
                elif mt == TX:
                    self._handle_tx(body)
        except (ConnectionError, OSError, ValueError):
            pass
        finally:
            with self.lock:
                self.peers.pop(peer_id, None)

    # ------------------------------------------------------------------
    # Message handlers
    # ------------------------------------------------------------------

    def _handle_announce(self, peer_id: bytes, tx_hash: bytes):
        with self.lock:
            if tx_hash in self.seen_tx:
                return
            conn = self.peers.get(peer_id)
        if conn is not None:
            try:
                send_msg(conn, encode_msg(GET_TX, tx_hash))
            except OSError:
                pass

    def _handle_tx(self, tx_bytes: bytes):
        h = keccak256(tx_bytes)
        with self.lock:
            if h in self.seen_tx:
                return
            self.seen_tx.add(h)
            self.tx_pool[h] = tx_bytes
        print(f"[{self.port}] got tx {h.hex()[:8]}...")
        self._gossip(h)

    # ------------------------------------------------------------------
    # Sqrt-fanout gossip
    # ------------------------------------------------------------------

    def _gossip(self, tx_hash: bytes):
        """Send TX body to sqrt(N) peers; ANNOUNCE_TX (hash only) to the rest."""
        with self.lock:
            peers = list(self.peers.items())
            body = self.tx_pool.get(tx_hash)
        if not peers or body is None:
            return
        k = max(1, int(math.sqrt(len(peers))))
        full_recipients = set(p[0] for p in random.sample(peers, min(k, len(peers))))
        for pid, conn in peers:
            try:
                if pid in full_recipients:
                    send_msg(conn, encode_msg(TX, body))
                else:
                    send_msg(conn, encode_msg(ANNOUNCE_TX, tx_hash))
            except OSError:
                pass

    def submit_tx(self, tx_bytes: bytes):
        """Inject a new transaction into this node and gossip it."""
        h = keccak256(tx_bytes)
        with self.lock:
            self.seen_tx.add(h)
            self.tx_pool[h] = tx_bytes
        self._gossip(h)


Overwriting _lib/peer.py


## 7. 8-node demo

Spin up 8 nodes on ports 5571-5578, let each one dial 3 random peers,
then verify every node has at least one connection.


In [3]:
import importlib, sys, time, random, secrets

# Ensure _lib is on the path
if '.' not in sys.path:
    import os
    sys.path.insert(0, os.path.abspath('.'))

import _lib.peer as _p
importlib.reload(_p)
from _lib.peer import Node
from _lib.keccak import keccak256

nodes = [Node(5570 + i) for i in range(1, 9)]
for n in nodes:
    n.start()
time.sleep(0.3)

# Each node connects to 3 random others
random.seed(0)
for n in nodes:
    others = [m for m in nodes if m is not n]
    for m in random.sample(others, 3):
        n.connect('127.0.0.1', m.port)

time.sleep(0.5)
peer_counts = [len(n.peers) for n in nodes]
print('\npeer counts:', peer_counts)
assert all(c > 0 for c in peer_counts), f'a node has no peers: {peer_counts}'
print('mesh formed OK')


[5578] inbound peer 5c2dd9ac9aaa3a72 from :5571
[5571] outbound peer b247a647e2476f94 -> :5578
[5575] inbound peer 5c2dd9ac9aaa3a72 from :5571
[5571] outbound peer a11527ec80f1c270 -> :5575
[5577] inbound peer 5c2dd9ac9aaa3a72 from :5571
[5571] outbound peer 7b67fb632db6a842 -> :5577
[5572] outbound peer 5c2dd9ac9aaa3a72 -> :5571
[5571] inbound peer bb21ddecba443ca1 from :5572
[5574] inbound peer bb21ddecba443ca1 from :5572
[5572] outbound peer d67933961e9261ef -> :5574
[5576] inbound peer bb21ddecba443ca1 from :5572
[5572] outbound peer 245aa3b7d5aeb4c2 -> :5576
[5575] inbound peer 750a016a309bba5b from :5573
[5573] outbound peer a11527ec80f1c270 -> :5575
[5578] inbound peer 750a016a309bba5b from :5573
[5573] outbound peer b247a647e2476f94 -> :5578
[5574] inbound peer 750a016a309bba5b from :5573
[5573] outbound peer d67933961e9261ef -> :5574
[5575] inbound peer d67933961e9261ef from :5574
[5574] outbound peer a11527ec80f1c270 -> :5575
[5576] inbound peer d67933961e9261ef from :5574
[5


peer counts: [5, 5, 5, 4, 7, 6, 4, 6]
mesh formed OK


In [4]:
test_tx = b'hand_built_tx_payload_' + secrets.token_bytes(8)
test_hash = keccak256(test_tx)
print('\nsubmitting tx', test_hash.hex()[:8], 'at node :', nodes[0].port)
nodes[0].submit_tx(test_tx)

# Poll up to 2 s for global saturation
have = []
for _ in range(20):
    have = [n for n in nodes if test_hash in n.tx_pool]
    if len(have) == len(nodes):
        break
    time.sleep(0.1)

print(f'\n{len(have)}/{len(nodes)} nodes have the tx')
assert len(have) == len(nodes), f'only {len(have)}/{len(nodes)} saturated'
print('saturation confirmed')



submitting tx 51eb22d4 at node : 5571
[5578] got tx 51eb22d4...
[5577] got tx 51eb22d4...
[5575] got tx 51eb22d4...
[5576] got tx 51eb22d4...
[5572] got tx 51eb22d4...
[5573] got tx 51eb22d4...
[5574] got tx 51eb22d4...

8/8 nodes have the tx
saturation confirmed


## 8. Observing frames with tcpdump

In a separate terminal, run:

```bash
sudo tcpdump -i lo -X 'tcp portrange 5571-5578' -c 30
```

Then re-run the submit cell above.  You will see:

* **HELLO frames** (`01` + node_id + port) during `connect()`
* **ANNOUNCE_TX frames** (`04` + 32-byte hash) — to non-full recipients
* **TX frames** (`06` + raw payload) — to the sqrt-chosen full recipients

bouncing between ports 5571-5578.


## 9. Debugging exercises

### 9a. Kill a node mid-gossip

P2P networks tolerate node failures.  Stop one node, then broadcast a new
transaction and confirm all *surviving* nodes still receive it.


In [5]:
print('\n--- killing node :', nodes[3].port, 'and re-submitting ---')
nodes[3].stop()
time.sleep(0.3)

test_tx2 = b'second_payload_' + secrets.token_bytes(8)
test_hash2 = keccak256(test_tx2)
nodes[0].submit_tx(test_tx2)

have2 = []
for _ in range(20):
    have2 = [n for n in nodes if n is not nodes[3] and test_hash2 in n.tx_pool]
    if len(have2) == len(nodes) - 1:
        break
    time.sleep(0.1)

print(f'after kill, {len(have2)}/{len(nodes)-1} surviving nodes have the new tx')
assert len(have2) == len(nodes) - 1, (
    f'expected {len(nodes)-1} surviving, got {len(have2)}'
)
print('kill-a-node resilience confirmed')



--- killing node : 5574 and re-submitting ---


[5578] got tx c90fca40...
[5577] got tx c90fca40...
[5573] got tx c90fca40...
[5572] got tx c90fca40...
[5575] got tx c90fca40...
[5576] got tx c90fca40...
after kill, 7/7 surviving nodes have the new tx
kill-a-node resilience confirmed


### 9b. Bandwidth comparison: sqrt-fanout vs full broadcast

Subclass `Node` to count bytes per gossip strategy.  The origin node does the bulk of the work, so we measure only its `bytes_sent`.

**Why sqrt-fanout matters:** for N peers, full broadcast sends N copies of the
full payload. Sqrt-fanout sends √N full copies plus (N-√N) 32-byte hashes.
For N=7 and a 80-byte payload that is ~7 full sends vs ~3 full + 4 hash sends —
already a win, and the gap widens dramatically with larger networks.


In [6]:
import math
from _lib.peer import encode_msg, TX, ANNOUNCE_TX
from _lib.framing import send_msg
from _lib.keccak import keccak256


class CountingNode(Node):
    def __init__(self, port, strategy):
        super().__init__(port)
        self.strategy = strategy  # 'sqrt' or 'broadcast'
        self.bytes_sent = 0

    def _gossip(self, tx_hash):
        with self.lock:
            peers = list(self.peers.items())
            body = self.tx_pool.get(tx_hash)
        if not peers or body is None:
            return
        if self.strategy == 'broadcast':
            for pid, conn in peers:
                frame = encode_msg(TX, body)
                self.bytes_sent += 4 + len(frame)  # 4-byte length header
                try:
                    send_msg(conn, frame)
                except OSError:
                    pass
        else:
            k = max(1, int(math.sqrt(len(peers))))
            full = set(p[0] for p in random.sample(peers, min(k, len(peers))))
            for pid, conn in peers:
                if pid in full:
                    frame = encode_msg(TX, body)
                else:
                    frame = encode_msg(ANNOUNCE_TX, tx_hash)
                self.bytes_sent += 4 + len(frame)
                try:
                    send_msg(conn, frame)
                except OSError:
                    pass


def build(strategy, base_port):
    ns = [CountingNode(base_port + i, strategy) for i in range(1, 9)]
    for n in ns:
        n.start()
    time.sleep(0.3)
    random.seed(42)
    for n in ns:
        others = [m for m in ns if m is not n]
        for m in random.sample(others, 3):
            n.connect('127.0.0.1', m.port)
    time.sleep(0.5)
    return ns


ns_sqrt = build('sqrt', 5580)
ns_bcast = build('broadcast', 5590)

payload = b'bandwidth_test_' + secrets.token_bytes(64)  # 79 bytes, realistic small tx
ns_sqrt[0].submit_tx(payload)
ns_bcast[0].submit_tx(payload)
time.sleep(1.0)

print(f'\norigin node bytes sent  sqrt-fanout : {ns_sqrt[0].bytes_sent}')
print(f'origin node bytes sent  full broadcast: {ns_bcast[0].bytes_sent}')
print('(sqrt fanout sends full bodies to ~sqrt(N) peers, hashes to the rest)')

assert ns_sqrt[0].bytes_sent < ns_bcast[0].bytes_sent, (
    'expected sqrt-fanout to use fewer bytes than broadcast'
)
print('bandwidth advantage confirmed')

for n in ns_sqrt:
    n.stop()
for n in ns_bcast:
    n.stop()


[5587] inbound peer a3ede30a00f65749 from :5581
[5581] outbound peer 55a51f94ec5e3fca -> :5587
[5582] inbound peer a3ede30a00f65749 from :5581
[5581] outbound peer 30c6abaec5094ea8 -> :5582
[5588] inbound peer a3ede30a00f65749 from :5581
[5581] outbound peer 93927e82c0aa5616 -> :5588
[5587] inbound peer 30c6abaec5094ea8 from :5582
[5582] outbound peer 55a51f94ec5e3fca -> :5587
[5584] inbound peer 30c6abaec5094ea8 from :5582
[5582] outbound peer 2ca4a979e83caaf9 -> :5584
[5583] inbound peer 30c6abaec5094ea8 from :5582
[5582] outbound peer 00fc0cae1f8bc232 -> :5583
[5588] inbound peer 00fc0cae1f8bc232 from :5583
[5583] outbound peer 93927e82c0aa5616 -> :5588
[5581] inbound peer 00fc0cae1f8bc232 from :5583
[5583] outbound peer a3ede30a00f65749 -> :5581
[5587] inbound peer 2ca4a979e83caaf9 from :5584
[5584] outbound peer 55a51f94ec5e3fca -> :5587
[5588] inbound peer 2ca4a979e83caaf9 from :5584
[5584] outbound peer 93927e82c0aa5616 -> :5588
[5586] inbound peer 2ca4a979e83caaf9 from :5584
[5

[5597] inbound peer 2534b80a47d187e8 from :5591
[5591] outbound peer 24dc62a14449e36d -> :5597
[5592] inbound peer 2534b80a47d187e8 from :5591
[5591] outbound peer 2ff283967009b307 -> :5592
[5598] inbound peer 2534b80a47d187e8 from :5591
[5591] outbound peer 94b5f2c123db0e9a -> :5598
[5597] inbound peer 2ff283967009b307 from :5592
[5592] outbound peer 24dc62a14449e36d -> :5597
[5594] inbound peer 2ff283967009b307 from :5592
[5592] outbound peer 00c1c6b001587157 -> :5594
[5593] inbound peer 2ff283967009b307 from :5592
[5592] outbound peer e47ad12e5c136635 -> :5593
[5598] inbound peer e47ad12e5c136635 from :5593
[5593] outbound peer 94b5f2c123db0e9a -> :5598
[5591] inbound peer e47ad12e5c136635 from :5593
[5593] outbound peer 2534b80a47d187e8 -> :5591
[5597] inbound peer 00c1c6b001587157 from :5594
[5594] outbound peer 24dc62a14449e36d -> :5597
[5598] inbound peer 00c1c6b001587157 from :5594
[5594] outbound peer 94b5f2c123db0e9a -> :5598
[5596] inbound peer 00c1c6b001587157 from :5594
[5

[5582] got tx 694f91ec...
[5585] got tx 694f91ec...
[5598] got tx 694f91ec...
[5586] got tx 694f91ec...
[5584] got tx 694f91ec...
[5592] got tx 694f91ec...
[5597] got tx 694f91ec...
[5588] got tx 694f91ec...
[5593] got tx 694f91ec...
[5594] got tx 694f91ec...
[5596] got tx 694f91ec...
[5583] got tx 694f91ec...
[5587] got tx 694f91ec...
[5595] got tx 694f91ec...



origin node bytes sent  sqrt-fanout : 316
origin node bytes sent  full broadcast: 504
(sqrt fanout sends full bodies to ~sqrt(N) peers, hashes to the rest)
bandwidth advantage confirmed


## 10. Cleanup

Stop all nodes from the 8-node demo (node 3 was already stopped in §9a).


In [7]:
for n in nodes:
    if n is not nodes[3]:
        n.stop()
print('all nodes stopped')


all nodes stopped
